# NGC 4151 Richardson–Lucy spectral deconvolution

This notebook adapts the COSIpy v0.2.1 Crab multi-energy image-deconvolution workflow to the supplied NGC 4151 DC4 files. It reconstructs Galactic sky maps, then performs a point-source Richardson–Lucy spectral unfolding at the known NGC 4151 position and displays the corresponding $E^2dN/dE$ SED.

The supplied source and background histograms are already accumulated in Galactic CDS coordinates. Consequently, the spacecraft orientation is used to build an exposure-integrated Galactic response rather than replaying the original ScAtt-binned workflow literally.

> The cutoff-power-law parameters are obtained with a COSI/3ML Poisson forward-folded fit over the complete 100–10,000 keV response. The non-parametric RL points are shown for visualization; weak high-energy bins should not be treated as Gaussian flux measurements.

In [1]:
from pathlib import Path
import json
import sys

import astropy.units as u
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from astromodels import Cutoff_powerlaw, Model, Parameter, PointSource
from astropy.coordinates import SkyCoord
from astropy.table import Table
from threeML import DataList, JointLikelihood

from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.data_io import EmCDSBinnedData
from cosipy.image_deconvolution import (
    build_galactic_response,
    prepare_galactic_histograms,
    run_richardson_lucy_spectral_deconvolution,
    select_orientation_for_pointing_cut,
)
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.response import (
    BinnedInstrumentResponse,
    BinnedThreeMLModelFolding,
    BinnedThreeMLPointSourceResponse,
    FullDetectorResponse,
    PointSourceResponse,
)
from cosipy.statistics import PoissonLikelihood

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

10:37:37 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=823618;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=977603;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  available                                                                                        

WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=85344;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=380539;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#65\65]8;;\
                  will not be available.                                                                           

WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=923824;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=907909;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

10:37:38 INFO      Starting 3ML!                                                                     ]8;id=393923;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=325392;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\

WARNING   WARNINGs here are NOT errors                                                      ]8;id=825103;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=602258;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#45\45]8;;\

...

WARNING   ROOT minimizer not available                                                ]8;id=72192;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=343503;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py#1208\1208]8;;\

WARNING   Multinest minimizer not available                                           ]8;id=858790;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=115605;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py#1218\1218]8;;\

WARNING   PyGMO is not available                                                      ]8;id=407477;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py\minimization.py]8;;\:]8;id=765415;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/minimizer/minimization.py#1228\1228]8;;\

WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=146540;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=224681;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

10:37:39 WARNING   No fermitools installed                                              ]8;id=763237;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=432796;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

## Configuration

The default nside-2 all-sky run is intentionally inexpensive and is suitable for validating the imaging workflow. Increase `NSIDE_IMAGE` to 4 or 8 for a better-resolved image; response size and construction cost scale with the number of sky pixels. `NSIDE_ALLSKY_SCATT_MAP=4` is used only for that coarse map. The point-source RL spectrum and 3ML fit use `NSIDE_SPECTRAL_SCATT_MAP=16`, matching the response sampling used to inject the source and in the Paper_Plots_DC4 analysis.

In [2]:
DATA_FILE = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/GammaRay/Paper_Models/"
    "NGC4151_ec_1000_DC4_COSI_cpl_60_fovCut.hdf5"
)
BACKGROUND_FILE = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/DC4_Files/Background/"
    "Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_"
    "NGC4151_60deg_fov_cut.hdf5"
)
DETECTOR_RESPONSE_FILE = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/DC4_Files/"
    "ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered."
    "nonsparse.binnedimaging.imagingresponse.h5"
)
ORIENTATION_FILE = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/DC4_Files/"
    "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
)

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "pyproject.toml").exists():
        REPOSITORY_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from inside the COSIpy repository.")

OUTPUT_DIR = REPOSITORY_ROOT / "outputs/ngc4151_rl_deconvolution"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LARGE_DATA_DIR = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/"
    "Radio_Quiet_AGN/SED-analysis"
)
LARGE_DATA_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_COORD = SkyCoord(l=155.077, b=75.063, unit="deg", frame="galactic")
FOV_CUT = 60 * u.deg
DATA_CONTAINS_BACKGROUND = False
NSIDE_IMAGE = 2
NSIDE_ALLSKY_SCATT_MAP = 4
# Match the source-injection and Paper_Plots_DC4 response sampling: 2 x nside(PsiChi=8).
NSIDE_SPECTRAL_SCATT_MAP = 16
ITERATIONS = 20
INITIAL_FLUX = 1e-4
SMOOTHING_FWHM = 3 * u.deg
SPECTRAL_ITERATIONS = 1000
INJECTED_NORMALIZATION = 0.15
INJECTED_INDEX = -1.75
INJECTED_CUTOFF_KEV = 1000.0
FIT_PIVOT_KEV = 200.0

## Load the data and select the orientation history

The background contains a time axis, while the source histogram is already time integrated. `prepare_galactic_histograms` projects out that background time axis and, by default, forms the observed histogram as source plus background.

In [3]:
source, background, event = prepare_galactic_histograms(
    DATA_FILE,
    BACKGROUND_FILE,
    data_contains_background=DATA_CONTAINS_BACKGROUND,
)
orientation = select_orientation_for_pointing_cut(
    ORIENTATION_FILE,
    SOURCE_COORD,
    FOV_CUT,
)

source_counts = float(source.to_dense(copy=False).contents.sum())
background_counts = float(background.to_dense(copy=False).contents.sum())
livetime = orientation.cumulative_livetime()

print(f"Source counts:     {source_counts:,.3f}")
print(f"Background counts: {background_counts:,.0f}")
print(f"Selected livetime: {livetime.to_value(u.s):,.0f} s")
print(f"CDS axes:          {event.axes.labels}")

Source counts:     177,963.933
Background counts: 25,071,034
Selected livetime: 980,415 s
CDS axes:          ['Em' 'Phi' 'PsiChi']


## Build or reuse the Galactic response

This step folds the detector response through the selected orientation history for every Galactic image pixel. Existing response files and cached pixel rows are reused automatically.

In [4]:
response_path = LARGE_DATA_DIR / (
    f"ngc4151_galactic_response_nside{NSIDE_IMAGE}_scatt{NSIDE_ALLSKY_SCATT_MAP}.hdf5"
)
response = build_galactic_response(
    DETECTOR_RESPONSE_FILE,
    orientation,
    response_path,
    nside_image=NSIDE_IMAGE,
    nside_scatt_map=NSIDE_ALLSKY_SCATT_MAP,
    earth_occ=True,
)
print(response_path)

/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/SED-analysis/ngc4151_galactic_response_nside2_scatt4.hdf5


## Accelerated multi-energy Richardson–Lucy reconstruction

The model has axes `(Galactic sky pixel, incident energy)`. The update uses MaxStep acceleration, exposure weighting, Gaussian smoothing, and simultaneous optimization of one background normalization.

In [5]:
algorithm = run_richardson_lucy_spectral_deconvolution(
    event,
    background,
    response,
    iteration_max=ITERATIONS,
    initial_flux=INITIAL_FLUX,
    acceleration_max=5.0,
    response_weighting_index=0.5,
    smoothing_fwhm=SMOOTHING_FWHM,
    background_range=(0.01, 10.0),
    stopping_threshold=0.01,
)
model = algorithm.results[-1]["model"]
model.write(OUTPUT_DIR / "reconstructed_model.hdf5", overwrite=True)
print(f"Completed {len(algorithm.results)} iterations")

Richardson-Lucy iterations:   0%|          | 0/20 [00:00<?, ?it/s]

Completed 20 iterations


In [6]:
likelihood = np.array([
    np.sum(result["log-likelihood"]) for result in algorithm.results
])
background_norm = np.array([
    result["background_normalization"]["background"]
    for result in algorithm.results
])
iteration_number = np.arange(1, len(algorithm.results) + 1)

figure, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(iteration_number, likelihood, marker="o", ms=3)
axes[0].set(xlabel="Iteration", ylabel="Poisson log-likelihood")
axes[1].plot(iteration_number, background_norm, marker="o", ms=3)
axes[1].set(xlabel="Iteration", ylabel="Background normalization")
plt.show()

print(f"Final background normalization: {background_norm[-1]:.6f}")
print(f"Likelihood monotonic: {np.all(np.diff(likelihood) >= 0)}")

Final background normalization: 1.000787
Likelihood monotonic: True



WARNING UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown



## Reconstructed maps

The red star marks NGC 4151. Dark pixels are zero-exposure or Earth-occulted regions. At nside 2 each HEALPix pixel is large, so this display validates localization but is not a publication-resolution image.

In [7]:
energy_edges = model.axes["Ei"].edges.to_value(u.keV)
figure, axes = plt.subplots(2, 5, figsize=(20, 8))
for energy_index, axis in enumerate(axes.flat):
    plt.axes(axis)
    hp.mollview(
        model.contents[:, energy_index].value,
        title=(
            f"{energy_edges[energy_index]:g}–"
            f"{energy_edges[energy_index + 1]:g} keV"
        ),
        unit=str(model.unit),
        hold=True,
    )
    hp.projscatter(
        SOURCE_COORD.galactic.l.deg,
        SOURCE_COORD.galactic.b.deg,
        lonlat=True,
        color="red",
        marker="*",
    )
plt.show()


WARNING UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown



## Point-source spectral deconvolution

Summing every pixel of a weak-source all-sky reconstruction promotes background residuals into source flux. For the SED we instead build the response at the exact NGC 4151 coordinate and apply the Richardson–Lucy/ML-EM update only to the ten incident-energy amplitudes. Its scattering map uses nside 16, matching the simulated source response. The supplied simulated background is fixed at normalization one.

In [8]:
point_response_path = (
    LARGE_DATA_DIR
    / f"ngc4151_exact_point_source_response_scatt{NSIDE_SPECTRAL_SCATT_MAP}.hdf5"
)
if point_response_path.exists():
    point_response = PointSourceResponse.open(point_response_path)
else:
    with FullDetectorResponse.open(DETECTOR_RESPONSE_FILE, dtype=np.float32) as detector:
        scatt_map = orientation.get_scatt_map(
            nside=NSIDE_SPECTRAL_SCATT_MAP,
            target_coord=SOURCE_COORD,
            earth_occ=True,
        )
        point_response = detector.get_point_source_response(
            coord=SOURCE_COORD, scatt_map=scatt_map
        )
    point_response.write(point_response_path, overwrite=True)

energy_edges = point_response.axes["Ei"].edges.to_value(u.keV)
energy_centers = point_response.axes["Ei"].centers.to(u.keV)
energy_widths = np.diff(energy_edges)
response_matrix = np.asarray(point_response.contents.value).reshape(10, -1)
event_counts = np.asarray(event.to_dense(copy=False).contents).reshape(-1)
background_model = np.asarray(background.to_dense(copy=False).contents).reshape(-1)
summed_response = response_matrix.sum(axis=1)

band_flux = np.full(10, 1e-4)  # integrated flux in each Ei bin
spectral_log_likelihood = []
for spectral_iteration in range(SPECTRAL_ITERATIONS):
    expectation = background_model + band_flux @ response_matrix
    data_ratio = np.divide(
        event_counts,
        expectation,
        out=np.zeros_like(event_counts),
        where=expectation > 0,
    )
    band_flux *= (response_matrix @ data_ratio) / summed_response
    expectation = background_model + band_flux @ response_matrix
    supported = expectation > 0
    spectral_log_likelihood.append(
        np.sum(
            event_counts[supported] * np.log(expectation[supported])
            - expectation[supported]
        )
    )

flux_values = band_flux / energy_widths
sed_values = energy_centers.to_value(u.keV) ** 2 * flux_values
injected_flux_values = (
    INJECTED_NORMALIZATION
    * energy_centers.to_value(u.keV) ** INJECTED_INDEX
    * np.exp(-energy_centers.to_value(u.keV) / INJECTED_CUTOFF_KEV)
)

spectrum_table = Table(
    {
        "e_min_keV": energy_edges[:-1],
        "e_max_keV": energy_edges[1:],
        "e_ref_keV": energy_centers.to_value(u.keV),
        "dnde_per_cm2_s_keV": flux_values,
        "e2dnde_keV_per_cm2_s": sed_values,
        "injected_dnde_per_cm2_s_keV": injected_flux_values,
    }
)
spectrum_table.write(
    OUTPUT_DIR / "reconstructed_spectrum_and_sed.csv",
    format="ascii.csv",
    overwrite=True,
)
spectrum_table

e_min_keV,e_max_keV,e_ref_keV,dnde_per_cm2_s_keV,e2dnde_keV_per_cm2_s,injected_dnde_per_cm2_s_keV
float64,float64,float64,float64,float64,float64
100.0,158.489,125.89241438625285,2.7813268020274526e-05,0.44080970352652904,2.7952306105915815e-05
158.489,251.189,199.52617227070735,1.156959564676876e-05,0.46059362529844716,1.1599475956943967e-05
251.189,398.107,316.22792290213715,4.602142202872706e-06,0.46021467692562634,4.6105723352986635e-06
398.107,630.957,501.1869894550339,1.7118657918444308e-06,0.4300008265274389,1.7117048665165975e-06
630.957,1000.0,794.328017886817,5.724851570457991e-07,0.36121351723414596,5.703207745740544e-07
1000.0,1584.89,1258.9241438625288,1.6219727756865653e-07,0.2570648432457882,1.6008410625674655e-07
1584.89,2511.89,1995.2617227070741,3.962509114807905e-08,0.15775023554753564,3.424209505557777e-08
2511.89,3981.07,3162.2792290213692,1.0520870191873429e-08,0.10520880630996443,4.7613538984606896e-09
3981.07,6309.57,5011.869894550336,5.566603643801907e-10,0.01398266253808638,3.3455243108521916e-10


## Full-range 3ML cutoff-power-law fit

Following the Crab spectral-fitting tutorial, we now build the standard COSI 3ML plugin. This fit is independent of the RL flux points: it folds a cutoff power law through the full response and evaluates the Poisson likelihood in every CDS bin from 100 to 10,000 keV. The point-source scattering map uses nside 16, matching the source injection and Paper_Plots_DC4 fit. The background rate is fitted as a nuisance parameter.

In [9]:
# Wrap the measured histogram and background for the COSI 3ML plugin.
fit_data = EmCDSBinnedData(event)
fit_background = background.to_dense(copy=True)
fit_background += sys.float_info.min  # Avoid log(0) in empty background bins.
background_fit = FreeNormBinnedBackground(
    fit_background,
    sc_history=orientation,
    copy=False,
)

# The 3ML background parameter is a rate rather than a scale factor.
background_rate = background_counts / livetime.to_value(u.s)

# Build the point-source response at the known NGC 4151 position.
detector = FullDetectorResponse.open(DETECTOR_RESPONSE_FILE, dtype=np.float32)
instrument_response = BinnedInstrumentResponse(detector, fit_data)
point_response_3ml = BinnedThreeMLPointSourceResponse(
    data=fit_data,
    instrument_response=instrument_response,
    sc_history=orientation,
    energy_axis=detector.axes["Ei"],
    polarization_axis=(
        detector.axes["Pol"] if "Pol" in detector.axes.labels else None
    ),
    nside=NSIDE_SPECTRAL_SCATT_MAP,
)
model_response = BinnedThreeMLModelFolding(
    data=fit_data,
    point_source_response=point_response_3ml,
)
poisson_likelihood = PoissonLikelihood(
    fit_data,
    model_response,
    background_fit,
)
cosi = ThreeMLPluginInterface(
    "cosi",
    poisson_likelihood,
    model_response,
    background_fit,
)
cosi.bkg_parameter["bkg_norm"] = Parameter(
    "bkg_norm",
    background_rate,
    min_value=0.0,
    max_value=5.0 * background_rate,
    delta=0.05 * background_rate,
    unit=u.Hz,
)

### Define the source model

`Cutoff_powerlaw` uses $K(E/\mathrm{piv})^{\mathrm{index}}\exp(-E/x_c)$. We fit at the same 200 keV pivot used by Paper_Plots_DC4. This greatly improves covariance conditioning compared with fitting the normalization at 1 keV; after fitting, $K$ is also converted back to the equivalent 1 keV normalization for comparison with the injected value.

In [10]:
spectrum = Cutoff_powerlaw()
spectrum.K.unit = 1 / (u.cm**2 * u.s * u.keV)
spectrum.piv.unit = u.keV
spectrum.xc.unit = u.keV

spectrum.K.bounds = (1e-8, 1e-2)
spectrum.K.value = 1e-5
spectrum.piv.value = FIT_PIVOT_KEV
spectrum.index.value = -1.75
spectrum.index.bounds = (-3.0, 1.0)
spectrum.xc.bounds = (100.0, 10000.0)
spectrum.xc.value = 1000.0

ngc4151 = PointSource(
    "NGC4151",
    l=SOURCE_COORD.galactic.l.deg,
    b=SOURCE_COORD.galactic.b.deg,
    spectral_shape=spectrum,
)
spectral_model = Model(ngc4151)

10:38:00 WARNING   We have set the min_value of Cutoff_powerlaw.K to 1e-99 because there was a     ]8;id=893897;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=720898;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#715\715]8;;\
                  postive transform                                                                                

WARNING   The current value of the parameter K (1.0) was above the new maximum 0.01.      ]8;id=553538;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=373386;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#810\810]8;;\

WARNING   We have set the min_value of Cutoff_powerlaw.xc to 1e-99 because there was a    ]8;id=457328;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=13193;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#715\715]8;;\
                  postive transform                                                                                

WARNING   The current value of the parameter xc (10.0) was below the new minimum 100.0.   ]8;id=129108;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=586437;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#732\732]8;;\

### Fit the model

`JointLikelihood.fit()` simultaneously fits the three source parameters and the background rate. The returned table includes parameter uncertainties and the correlation matrix.

In [11]:
plugins = DataList(cosi)
likelihood_analysis = JointLikelihood(
    spectral_model,
    plugins,
    verbose=False,
)
# Make the covariance samples used below reproducible.
np.random.seed(0)
fit_results, fit_statistics = likelihood_analysis.fit()

fitted_k_at_pivot = spectrum.K.value
fitted_index = spectrum.index.value
fitted_cutoff = spectrum.xc.value
fitted_norm = fitted_k_at_pivot * FIT_PIVOT_KEV ** (-fitted_index)
fitted_background_rate = cosi.bkg_parameter["bkg_norm"].value

parameter_prefix = "NGC4151.spectrum.main.Cutoff_powerlaw"
norm_fit = fit_results.loc[f"{parameter_prefix}.K"]
index_fit = fit_results.loc[f"{parameter_prefix}.index"]
cutoff_fit = fit_results.loc[f"{parameter_prefix}.xc"]

fit_results

10:38:00 INFO      set the minimizer to minuit                                             ]8;id=401202;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=404043;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

10:38:10 WARNING   The current value of the parameter K (1.0) was above the new maximum 0.01.      ]8;id=236795;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=829225;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#810\810]8;;\

WARNING   The current value of the parameter xc (10.0) was below the new minimum 100.0.   ]8;id=862318;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py\parameter.py]8;;\:]8;id=88941;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/core/parameter.py#732\732]8;;\

Best fit values:

,result,unit
parameter,,
NGC4151.spectrum.main.Cutoff_powerlaw.K,(1.41 +/- 0.09) x 10^-5,1 / (keV s cm2)
NGC4151.spectrum.main.Cutoff_powerlaw.index,-1.75 +/- 0.12,
NGC4151.spectrum.main.Cutoff_powerlaw.xc,(1.00 -0.26 +0.35) x 10^3,keV
bkg_norm,(2.5572 +/- 0.0008) x 10,Hz


...

Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi,-112959376.80830295
total,-112959376.80830295


Values of statistical measures:

,statistical measures
AIC,-225918745.61643228
BIC,-225918704.22631708


,value,negative_error,positive_error,error,unit
NGC4151.spectrum.main.Cutoff_powerlaw.K,0.000014,-8.432703e-07,9.157179e-07,8.794941e-07,1 / (keV s cm2)
NGC4151.spectrum.main.Cutoff_powerlaw.index,-1.750346,-1.175253e-01,1.227194e-01,1.201223e-01,
NGC4151.spectrum.main.Cutoff_powerlaw.xc,1000.857681,-2.622155e+02,3.444491e+02,3.033323e+02,keV
bkg_norm,25.571852,-7.621770e-03,7.546846e-03,7.584308e-03,Hz


## Photon spectrum and SED

In [12]:
plot_energy = np.geomspace(energy_edges[0], energy_edges[-1], 400)
fit_curve = fitted_k_at_pivot * (plot_energy / FIT_PIVOT_KEV)**fitted_index * np.exp(
    -plot_energy / fitted_cutoff
)
# Propagate the fitted covariance, including parameter correlations.
fit_samples = likelihood_analysis.results
pivot_normalization_samples = fit_samples.get_variates(
    f"{parameter_prefix}.K"
).samples
index_samples = fit_samples.get_variates(
    f"{parameter_prefix}.index"
).samples
cutoff_samples = fit_samples.get_variates(
    f"{parameter_prefix}.xc"
).samples
norm_at_1kev_samples = (
    pivot_normalization_samples * FIT_PIVOT_KEV ** (-index_samples)
)
norm_at_1kev_lower, norm_at_1kev_upper = np.percentile(
    norm_at_1kev_samples, [16, 84]
)
sampled_sed = (
    pivot_normalization_samples[:, None]
    * (plot_energy[None, :] / FIT_PIVOT_KEV) ** index_samples[:, None]
    * plot_energy[None, :] ** 2
    * np.exp(-plot_energy[None, :] / cutoff_samples[:, None])
)
sed_lower, sed_upper = np.percentile(sampled_sed, [16, 84], axis=0)

injected_curve = (
    INJECTED_NORMALIZATION
    * plot_energy**INJECTED_INDEX
    * np.exp(-plot_energy / INJECTED_CUTOFF_KEV)
)
xerr = np.vstack((
    energy_centers.to_value(u.keV) - energy_edges[:-1],
    energy_edges[1:] - energy_centers.to_value(u.keV),
))

figure, axis = plt.subplots(figsize=(6.5, 5), constrained_layout=True)
axis.errorbar(
    energy_centers.to_value(u.keV),
    sed_values,
    xerr=xerr,
    fmt="o",
    label=r"Spectral Deconvolution SED",
)
axis.fill_between(
    plot_energy,
    sed_lower,
    sed_upper,
    alpha=0.25,
    label="3ML 68% uncertainty",
)
axis.plot(plot_energy, plot_energy**2 * fit_curve, label="3ML forward fit")
axis.plot(
    plot_energy, plot_energy**2 * injected_curve, "--", label="Injected CPL"
)
axis.set(
    xscale="log",
    yscale="log",
    xlabel="Energy (keV)",
    ylabel=r"$E^2dN/dE$ (keV cm$^{-2}$ s$^{-1}$)",
)
pivot_norm_scale = 1e-5
fit_text = "\n".join((
    rf"$K_{{200}}=({fitted_k_at_pivot / pivot_norm_scale:.2f}_{{-{abs(norm_fit['negative_error']) / pivot_norm_scale:.2f}}}^{{+{norm_fit['positive_error'] / pivot_norm_scale:.2f}}})\times10^{{-5}}$",
    rf"$\Gamma={fitted_index:.3f}_{{-{abs(index_fit['negative_error']):.3f}}}^{{+{index_fit['positive_error']:.3f}}}$",
    rf"$E_c={fitted_cutoff:.0f}_{{-{abs(cutoff_fit['negative_error']):.0f}}}^{{+{cutoff_fit['positive_error']:.0f}}}\ \mathrm{{keV}}$",
))
axis.text(
    0.03,
    0.04,
    fit_text,
    transform=axis.transAxes,
    fontsize=8,
    bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "0.8"},
)
axis.legend(fontsize=8)
figure.savefig(OUTPUT_DIR / "reconstructed_sed_with_fit.png", dpi=180)
plt.show()


WARNING UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown



In [13]:
diagnostics = {
    "iterations": len(algorithm.results),
    "livetime_s": livetime.to_value(u.s),
    "source_counts": source_counts,
    "background_counts": background_counts,
    "final_log_likelihood": float(likelihood[-1]),
    "final_background_normalization": float(background_norm[-1]),
    "spectral_rl_iterations": SPECTRAL_ITERATIONS,
    "allsky_scatt_map_nside": NSIDE_ALLSKY_SCATT_MAP,
    "spectral_scatt_map_nside": NSIDE_SPECTRAL_SCATT_MAP,
    "fit_energy_min_keV": float(energy_edges[0]),
    "fit_energy_max_keV": float(energy_edges[-1]),
    "fit_method": "COSI/3ML JointLikelihood",
    "fit_pivot_keV": FIT_PIVOT_KEV,
    "fit_norm_at_pivot": float(fitted_k_at_pivot),
    "fit_norm_at_pivot_error_low": float(abs(norm_fit["negative_error"])),
    "fit_norm_at_pivot_error_high": float(norm_fit["positive_error"]),
    "fit_norm_at_1keV": float(fitted_norm),
    "fit_norm_error_low": float(fitted_norm - norm_at_1kev_lower),
    "fit_norm_error_high": float(norm_at_1kev_upper - fitted_norm),
    "fit_photon_index": float(fitted_index),
    "fit_index_error_low": float(abs(index_fit["negative_error"])),
    "fit_index_error_high": float(index_fit["positive_error"]),
    "fit_cutoff_keV": float(fitted_cutoff),
    "fit_cutoff_error_low_keV": float(abs(cutoff_fit["negative_error"])),
    "fit_cutoff_error_high_keV": float(cutoff_fit["positive_error"]),
    "fit_background_rate_hz": float(fitted_background_rate),
}
(OUTPUT_DIR / "notebook_diagnostics.json").write_text(
    json.dumps(diagnostics, indent=2)
)
print(f"Results written to {OUTPUT_DIR}")
diagnostics

Results written to /Users/parshadkp/Software/cosipy/outputs/ngc4151_rl_deconvolution


{'iterations': 20,
 'livetime_s': np.float64(980415.0),
 'source_counts': 177963.9331755011,
 'background_counts': 25071034.0,
 'final_log_likelihood': 112958586.22928773,
 'final_background_normalization': 1.0007866127277436,
 'spectral_rl_iterations': 1000,
 'allsky_scatt_map_nside': 4,
 'spectral_scatt_map_nside': 16,
 'fit_energy_min_keV': 100.0,
 'fit_energy_max_keV': 10000.0,
 'fit_method': 'COSI/3ML JointLikelihood',
 'fit_pivot_keV': 200.0,
 'fit_norm_at_pivot': 1.4101415142736186e-05,
 'fit_norm_at_pivot_error_low': 8.432703318185134e-07,
 'fit_norm_at_pivot_error_high': 9.157178656750466e-07,
 'fit_norm_at_1keV': 0.15026652588113673,
 'fit_norm_error_low': 0.06870201796440974,
 'fit_norm_error_high': 0.11883776973476043,
 'fit_photon_index': -1.7503463905439411,
 'fit_index_error_low': 0.11752530253611071,
 'fit_index_error_high': 0.12271935660647548,
 'fit_cutoff_keV': 1000.8576810007494,
 'fit_cutoff_error_low_keV': 262.21552606819057,
 'fit_cutoff_error_high_keV': 344.4490

## Interpretation and next steps

The all-sky maps validate localization, while the point-source RL unfolding provides non-parametric SED points at the known source position. Their horizontal bars show energy-bin widths; they do not have Gaussian flux uncertainties. The independent COSI/3ML fit uses the complete 100–10,000 keV CDS, so weak high-energy bins contribute as Poisson nondetections rather than as precise positive flux measurements. The 3ML table reports local parameter uncertainties and correlations; simulation ensembles are still needed to establish coverage for this weak-source analysis.